# 01. Pengumpulan Data (*Web Scraping*) dan Kompilasi Korpus Berita Arab

Notebook ini menjalankan **Tahap 1 metodologi penelitian** mulai dari spesifikasi *web scraping* 6 portal berita daring berbahasa Arab (*Alsumaria*, *Al-Sharq*, *Assahraa*, *Akhbar El-Syaha*, *Saraya News*, dan *Addustour*), demonstrasi parsing HTML menjadi berkas teks & DataFrame, hingga audit kompilasi dataset utama `data/basic_articles_dataset.csv` dan penyimpanan ringkasan statistiknya ke `results/stats/`.

- **Pendekatan Medan Makna (Artikel Jurnal)**: *Studies in English Language and Education*, 11(3), 1686–1709, 2024 ([DOI: 10.24815/siele.v11i3.38090](https://doi.org/10.24815/siele.v11i3.38090)).
- **Pendekatan Semantik *Frame* (Tesis Magister)**: UIN Sunan Kalijaga Yogyakarta, 2024 ([Repositori UIN](https://digilib.uin-suka.ac.id/id/eprint/75147/)).


In [ ]:
import sys
from pathlib import Path

# Resolve root repositori secara relatif
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "src" / "arabic_news").exists() else cwd.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT.name}")

import json
import tempfile
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from arabic_news.acquisition import (
    PORTAL_SCRAPER_CONFIG,
    compile_raw_articles_dataset,
    compute_corpus_distributions,
    parse_category_urls_html,
    scrape_article_html,
)
from arabic_news.config import (
    FIGURES_DIR,
    LABELS,
    LABELS_ID,
    STATS_DIR,
    resolve_data_file,
)
from arabic_news.preprocessing import reshape_arabic_for_plot

plt.rcParams["figure.dpi"] = 120


## 1.1 Spesifikasi *Web Scraper* untuk 6 Portal Berita Arab

Proses pengumpulan data terdiri atas dua langkah:
1. **Crawler URL Indeks Kategori (`fetch_category_article_urls`)**: menelusuri halaman indeks kategori untuk menghimpun daftar URL berita.
2. **Ekstraktor Konten HTML (`scrape_article_html`)**: mengambil judul dan paragraf isi artikel dari DOM HTML portal lalu menyimpannya sebagai berkas `<index>.txt` di struktur direktori `sources/<kategori>/<portal>/`.


In [ ]:
scraper_rows = []
for portal_key, cfg in PORTAL_SCRAPER_CONFIG.items():
    scraper_rows.append({
        "Kode Portal": portal_key,
        "Domain Website": cfg["domain"],
        "Asal Negara": cfg["country"],
        "Selektor URL Artikel": cfg["link_selector"],
        "Selektor Judul": cfg["title_selector"],
        "Selektor Isi Berita": cfg["content_selector"],
        "Pola URL Indeks": cfg["url_template"],
    })

display(pd.DataFrame(scraper_rows))

print("Daftar 11 Kelas Kategori Berita Arab:")
display(pd.DataFrame([
    {"Label ID": cid, "Kategori (Arab)": cname, "Terjemahan Indonesia": LABELS_ID[cid]}
    for cid, cname in LABELS.items()
]))


## 1.2 Simulasi Alur *Parsing* HTML ke Berkas Teks & Kompilasi DataFrame

Menguji fungsi `parse_category_urls_html()`, `scrape_article_html()`, dan `compile_raw_articles_dataset()` menggunakan contoh DOM HTML berita olahraga Arab secara *offline* (tanpa bergantung pada berkas luar).


In [ ]:
sample_html_index = (
    '<div class="ec-arbic">'
    '<a class="title" href="/article/sample-sport-1">فوز المنتخب في البطولة</a>'
    '</div>'
)
sample_html_article = (
    '<div class="minit--1">'
    '<h1 class="title">فوز المنتخب الوطني في البطولة الدولية للكرة الطائرة</h1>'
    '<p>حقق المنتخب الوطني فوزا مهما في منافسات الجولة الأولى من التصفيات التمهيدية للبطولة.</p>'
    '<p>وجاء الفوز بثلاثة أشواط نظيفة ليرفع الفريق رصيده من النقاط في جدول الترتيب.</p>'
    '</div>'
)

extracted_urls = parse_category_urls_html(sample_html_index, portal="alsharq")
parsed_title, parsed_body = scrape_article_html(sample_html_article, portal="alsharq")

with tempfile.TemporaryDirectory() as tmp_dir:
    target_folder = Path(tmp_dir) / "sources" / "رياضة" / "alsharq"
    target_folder.mkdir(parents=True)
    (target_folder / "1.txt").write_text(f"{parsed_title}\n\n{parsed_body}\n", encoding="utf-8")
    compiled_sample_df = compile_raw_articles_dataset(Path(tmp_dir) / "sources")

print("URL terekstrak (alsharq):", extracted_urls)
print("Judul terekstrak:", parsed_title)
display(compiled_sample_df[["label", "category", "filename", "source"]].assign(
    panjang_karakter=compiled_sample_df["text"].str.len(),
    jumlah_kata=compiled_sample_df["text"].str.split().str.len(),
))


## 1.3 Komputasi Distribusi Korpus dari `basic_articles_dataset.csv` & Simpan ke `results/`

Membaca satu berkas korpus utama `data/basic_articles_dataset.csv`, menghitung jumlah baris mentah, baris teks kosong (`NaN`), jumlah artikel valid, serta distribusi 11 kategori dan 6 portal berita, lalu menyimpannya ke `results/stats/` dan `results/figures/`.


In [ ]:
basic_csv = resolve_data_file("basic_articles_dataset.csv")
if not basic_csv.exists():
    raise FileNotFoundError(
        "Letakkan basic_articles_dataset.csv di folder data/ sebelum menjalankan notebook ini."
    )

df_basic = pd.read_csv(basic_csv, low_memory=False)
dists = compute_corpus_distributions(df_basic)
cat_df = dists["category_distribution"]
src_df = dists["source_distribution"]

print(f"Skema kolom dataset      : {df_basic.columns.tolist()}")
print(f"Total baris mentah       : {dists['total_raw_rows']:,}")
print(f"Total baris teks kosong  : {dists['empty_text_rows']:,}")
print(f"Total artikel valid      : {dists['total_valid_articles']:,}")
display(cat_df)
display(src_df)

# Simpan keluaran Tahap 1 ke results/stats/ dan results/figures/
STATS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

article_lines = ["Jumlah artikel perkategori:"] + [
    f"{row['Kategori']}: {int(row['Artikel Mentah'])}" for _, row in cat_df.iterrows()
]
(STATS_DIR / "[count]_article.txt").write_text("\n".join(article_lines) + "\n", encoding="utf-8")

dist_payload = {
    "total_raw_rows": dists["total_raw_rows"],
    "empty_text_rows": dists["empty_text_rows"],
    "total_valid_articles": dists["total_valid_articles"],
    "category_distribution": cat_df.to_dict(orient="records"),
    "source_distribution": src_df.to_dict(orient="records"),
}
(STATS_DIR / "corpus_distributions.json").write_text(
    json.dumps(dist_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ar_labels = [reshape_arabic_for_plot(c) for c in cat_df["Kategori"]]
axes[0].bar(ar_labels, cat_df["Artikel Valid"], color="steelblue")
axes[0].set_title(f"Distribusi Artikel per Kategori (Total: {dists['total_valid_articles']:,})")
axes[0].set_ylabel("Jumlah Artikel")
axes[0].tick_params(axis="x", rotation=35)
for idx, val in enumerate(cat_df["Artikel Valid"]):
    axes[0].text(idx, val + 250, f"{val:,}", ha="center", va="bottom", fontsize=8)

src_labels = [f"{r['Portal Berita']} ({r['Asal Negara']})\n{r['Jumlah Artikel']:,}" for _, r in src_df.iterrows()]
axes[1].pie(src_df["Jumlah Artikel"], labels=src_labels, autopct="%1.1f%%", startangle=140)
axes[1].set_title(f"Distribusi Artikel per Portal Berita (Total: {src_df['Jumlah Artikel'].sum():,})")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "corpus_distributions.png", dpi=150)
plt.show()
print("Output tersimpan ke results/stats/ dan results/figures/corpus_distributions.png")
